# 03 — The complete dataset: run all 417 farms and look at every result

Runs (or loads) the full Task 1 mapping step: **every registered farm with a tile**, through the same `run_pipeline()` that `scripts/map_buildings.py` uses. Everything after section 1 reads the saved run, so you can re-run any view without re-running the model.

| Section | What you see |
|---|---|
| 1 | Run the whole pipeline (≈9 min on this laptop), or load the latest run |
| 2 | Headline numbers, by species |
| 3 | Interactive statewide map: every building, plus farms where nothing was found |
| 4 | Building sizes, buildings per farm, floor area vs. registry headcount |
| 5 | Why candidates were thrown out |
| 6 | **Gallery: every farm with its boxes**, filterable by species/county, page by page |
| 7 | Farms with no detected building, the ones to check first |
| 8 | Comparison with the old committed detection file |
| 9 | Google Earth |

In [ ]:
import sys, json, subprocess, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
%matplotlib inline
%load_ext autoreload
%autoreload 2   # pick up edits to geo_anom/ without restarting the kernel
pd.set_option("display.width", 200, "display.max_columns", 30, "display.max_rows", 200)

from geo_anom.task1 import tiles, viz
from geo_anom.task1.pipeline import run_pipeline, latest_run
from geo_anom.task1.species import GROUPS, species_group

## 1. Run everything, or load the latest run

* `RUN_FULL = True` runs all 417 farms now (≈9 minutes, progress bar below) into `data/processed/task1/run_<today>/`.
* `RUN_FULL = False` loads the most recent complete run. Every section below works the same either way.

In [ ]:
RUN_FULL = False          # <- True to re-run the whole state now (~9 min)

import datetime as dt
if RUN_FULL:
    from tqdm import tqdm
    RUN = ROOT / f"data/processed/task1/run_{dt.date.today().isoformat()}"
    manifest = tiles.load_manifest()
    meta = run_pipeline(manifest, RUN, manifest=manifest,
                        progress=lambda xs: tqdm(xs, desc="farms", unit="farm"))
else:
    RUN = latest_run()
print("Run folder:", RUN.relative_to(ROOT))

In [ ]:
meta = json.loads((RUN / "run_meta.json").read_text())
sites = json.loads((RUN / "sites.json").read_text())
farms = pd.read_csv(RUN / "farms.csv")
farms["group"] = farms["animal_type"].map(species_group)
buildings = gpd.read_file(RUN / "buildings.geojson")
cand_path = RUN / "candidates.geojson"
candidates = gpd.read_file(cand_path) if cand_path.exists() else None
site_by_id = {tiles.site_id(s): s for s in sites}
print(f"run at {meta['run_at']} · code {meta['git_commit']} · model {meta['checkpoint_sha256']} · device {meta['device']}")

## 2. Headline numbers

In [ ]:
display(Markdown(f"""
| | |
|---|---|
| Farms processed | **{meta['tiles']}** |
| Shapes the model proposed | {meta['raw_candidates']:,} |
| Kept by the shape rule (per photo) | {meta['kept_detections']:,} |
| **Unique buildings** (barns seen in two overlapping photos counted once) | **{meta['unique_buildings']:,}** |
| Farms with ≥1 building | {(farms.buildings > 0).sum()} of {len(farms)} |
"""))
by_species = farms.groupby("group").agg(
    farms=("site_id", "size"),
    farms_with_buildings=("buildings", lambda s: int((s > 0).sum())),
    buildings=("buildings", "sum"),
    floor_area_ha=("building_area_m2", lambda s: round(s.sum() / 1e4, 1)),
    registry_headcount=("headcount", "sum"))
by_species["% farms found"] = (100 * by_species.farms_with_buildings / by_species.farms).round(0)
by_species.index = [GROUPS[g][0] for g in by_species.index]
by_species.sort_values("farms", ascending=False)

In [ ]:
county = farms.groupby("county").agg(farms=("site_id", "size"), buildings=("buildings", "sum"),
                                     no_building=("buildings", lambda s: int((s == 0).sum())))
ax = county.sort_values("buildings").plot.barh(y=["buildings"], figsize=(9, 6), legend=False, color="#2c8c3c")
ax.set_xlabel("unique buildings"); ax.set_title("Detected buildings by county")
for i, (_, r) in enumerate(county.sort_values("buildings").iterrows()):
    ax.text(r.buildings + 5, i, f"{r.farms} farms, {r.no_building} with none", va="center", fontsize=9)
plt.tight_layout()

## 3. Statewide map

Every building is a dot coloured by the permit's animal type. Red pins are registered farms where **nothing** was detected. Toggle layers top right; hover for details. The basemap needs internet; the static version below doesn't.

In [ ]:
viz.state_map(buildings, farms, sites)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 9))
cent = buildings.to_crs("EPSG:32618").centroid.to_crs("EPSG:4326")
for g, (label, rgb) in GROUPS.items():
    sel = (buildings["species_group"] == g).values
    if sel.any():
        ax.scatter(cent.x[sel], cent.y[sel], s=6, color=np.array(rgb) / 255, label=f"{label} ({sel.sum()})")
z = farms[farms.buildings == 0]
zs = [site_by_id[i] for i in z.site_id if i in site_by_id]
ax.scatter([s["lon"] for s in zs], [s["lat"] for s in zs], marker="x", color="red", s=40,
           label=f"Registered farm, nothing detected ({len(zs)})")
ax.set_aspect(1 / np.cos(np.radians(38.8))); ax.legend(loc="lower left", fontsize=9)
ax.set_title("All detected buildings (static)"); ax.set_xlabel("lon"); ax.set_ylabel("lat");

## 4. What the buildings look like, in numbers

Poultry houses cluster tightly (about 12–20 m wide, 100–180 m long). Anything far outside that is worth a look in the gallery.

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(16, 4))
for ax, col, bins in zip(axs, ["length_m", "width_m", "area_m2"], [40, 30, 40]):
    for g in buildings["species_group"].value_counts().index[:4]:
        ax.hist(buildings.loc[buildings.species_group == g, col], bins=bins, alpha=0.6,
                color=np.array(GROUPS[g][1]) / 255, label=GROUPS[g][0])
    ax.set_title(col); ax.set_yscale("log")
axs[0].legend(fontsize=8); plt.tight_layout()

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(15, 5))
farms.buildings.plot.hist(bins=range(0, int(farms.buildings.max()) + 2), ax=axs[0], color="#2c8c3c")
axs[0].set_title("Buildings per farm"); axs[0].set_xlabel("buildings")

p = farms[(farms.group == "broiler") & (farms.buildings > 0) & (farms.headcount > 0)]
axs[1].scatter(p.headcount, p.building_area_m2, s=12, color="#2c8c3c")
axs[1].set_xscale("log"); axs[1].set_yscale("log")
r = np.corrcoef(np.log(p.headcount), np.log(p.building_area_m2))[0, 1]
axs[1].set_title(f"Broiler farms: detected floor area vs registry headcount (log-log R² = {r**2:.2f}, n = {len(p)})")
axs[1].set_xlabel("registry headcount (permitted)"); axs[1].set_ylabel("detected floor area (m²)")
plt.tight_layout()

## 5. Why candidates were thrown out

Every shape the model proposed and the shape rule rejected, by the **first** rule it failed. A big "length/area too small" bar often means partial outlines of real houses, not junk. Check some in notebook 01.

In [ ]:
if candidates is None:
    print("candidates.geojson not in this run folder (it isn't committed to git); re-run with RUN_FULL = True to get it.")
else:
    rej = candidates[~candidates["kept"].astype(bool)]
    first = rej["reject_reasons"].fillna("").str.split(";").str[0].str.split(" ").str[0]
    display(Markdown(f"**{len(rej):,} rejected** of {len(candidates):,} candidates."))
    ax = first.value_counts().plot.barh(figsize=(7, 3), color="#c0392b")
    ax.set_xlabel("candidates"); ax.set_title("First failed criterion")
    plt.tight_layout(); plt.show()
    display(rej.groupby(rej["species_group"])["reject_reasons"].count().rename("rejected by species"))

## 6. Gallery: every farm with its boxes

Coloured fill = buildings assigned to *this* farm (colour = animal type). Thin white = a neighbour's buildings in the same photo. Yellow × = where the registry says the farm is (often hundreds of metres off).

Set the filters and page through. `PAGE = 0, 1, 2, …`; or `ALL_PAGES = True` to draw everything (slow for all 417).

In [ ]:
SPECIES = None         # e.g. "broiler", "dairy", "beef", "swine", "poultry_other", "unknown"; None = all
COUNTY = None          # e.g. "Somerset"; None = all
SORT_BY = "buildings"  # or "headcount", "farm_name", "raw_candidates"
PER_PAGE = 16
PAGE = 0
ALL_PAGES = False

sel = farms.copy()
if SPECIES: sel = sel[sel.group == SPECIES]
if COUNTY:  sel = sel[sel.county == COUNTY]
sel = sel.sort_values(SORT_BY, ascending=SORT_BY == "farm_name")
pages = max(int(np.ceil(len(sel) / PER_PAGE)), 1)
display(Markdown(f"**{len(sel)} farms** match · {pages} pages of {PER_PAGE}"))

for pg in (range(pages) if ALL_PAGES else [PAGE]):
    chunk = sel.iloc[pg * PER_PAGE:(pg + 1) * PER_PAGE]
    viz.gallery([site_by_id[i] for i in chunk.site_id], buildings, title=f"Page {pg + 1}/{pages}")
    plt.show()

## 7. Farms with no detected building

Each one is either a real miss, a registry location error (point far from the barns), or a farm that isn't there anymore. These are the first farms to look at by eye.

In [ ]:
zero = farms[farms.buildings == 0].sort_values(["group", "headcount"], ascending=[True, False])
display(zero[["farm_name", "county", "animal_type", "headcount", "raw_candidates", "kept_in_tile"]])
viz.gallery([site_by_id[i] for i in zero.site_id], buildings, title="No building detected")
plt.show()

## 8. Comparison with the old committed file

`data/processed/detections/full_registry_unet_tulbure_detections.geojson` was the basis of earlier reported numbers. It counted each photo's detections separately (so shared barns twice) and skipped 15 farms. Per farm, kept-in-photo counts should match exactly except those 15.

In [ ]:
old = gpd.read_file(ROOT / "data/processed/detections/full_registry_unet_tulbure_detections.geojson")
cmp_ = pd.DataFrame({"old_file": old.groupby("farm_name").size(),
                     "this_run": farms.groupby("farm_name").kept_in_tile.sum()}).fillna(0).astype(int)
cmp_["diff"] = cmp_.this_run - cmp_.old_file
display(Markdown(f"Old file: **{len(old):,}** features · this run: **{meta['kept_detections']:,}** kept per photo → "
                 f"**{meta['unique_buildings']:,}** unique. Farms identical: {(cmp_['diff'] == 0).sum()} / {len(cmp_)}."))
cmp_[cmp_["diff"] != 0].sort_values("diff", ascending=False)

## 9. Open the whole state in Google Earth

In [ ]:
print("KMZ:", (RUN / "buildings.kmz").relative_to(ROOT))
OPEN_IN_GOOGLE_EARTH = False
if OPEN_IN_GOOGLE_EARTH:
    subprocess.run(["open", "-a", "Google Earth Pro", str(RUN / "buildings.kmz")])